# 02 — Data Preprocessing

This notebook converts the raw GDC case-level export into a clean survival analysis table.

The main tasks are:

- load the raw cohort table
- parse and flatten nested `diagnoses`
- inspect repeated diagnosis rows per patient
- construct survival time and event indicator
- standardize core covariates
- save a cleaned analysis dataset for downstream modeling

In [76]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path

raw_path = Path("../data/raw/tcga-brca_cases_raw.csv")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

## Load raw data

In [77]:
df = pd.read_csv(raw_path)
print(df.shape)
df.head()

(1098, 10)


,id,case_id,submitter_id,diagnoses,project.project_id,demographic.race,demographic.gender,demographic.ethnicity,demographic.vital_status,demographic.days_to_death
0,e3935ce4-64d3-4a66-ba11-d308b844b410,e3935ce4-64d3-4a66-ba11-d308b844b410,TCGA-E9-A5FL,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,white,female,not hispanic or latino,Alive,NaN
1,e3b555aa-7f0a-49c6-9b13-182c61a144c1,e3b555aa-7f0a-49c6-9b13-182c61a144c1,TCGA-A2-A1G4,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,white,female,not hispanic or latino,Alive,NaN
2,17ca61a2-607a-45ff-88fa-ef72e80bf891,17ca61a2-607a-45ff-88fa-ef72e80bf891,TCGA-BH-A0HF,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,white,female,not reported,Alive,NaN
3,7d681cc6-689d-41c8-9e84-e13733089ec9,7d681cc6-689d-41c8-9e84-e13733089ec9,TCGA-AR-A1AS,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,asian,female,not reported,Alive,NaN
4,17f275c1-a0d4-487d-8f02-ea279584b4cd,17f275c1-a0d4-487d-8f02-ea279584b4cd,TCGA-D8-A13Y,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",TCGA-BRCA,white,female,not hispanic or latino,Alive,NaN


## Inspect raw columns

In [78]:
df.columns.tolist()

['id',
 'case_id',
 'submitter_id',
 'diagnoses',
 'project.project_id',
 'demographic.race',
 'demographic.gender',
 'demographic.ethnicity',
 'demographic.vital_status',
 'demographic.days_to_death']

## Parse the nested `diagnoses` field

The GDC API returns diagnosis information as nested JSON-like content. After saving to CSV, this usually appears as a string. We convert it back into Python objects before flattening.

In [79]:
def parse_maybe_literal(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return [x]
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            if isinstance(parsed, dict):
                return [parsed]
            return []
        except Exception:
            return []
    return []

## Check one parsed example

In [80]:
df["diagnoses_parsed"] = df["diagnoses"].apply(parse_maybe_literal)
df[["case_id", "diagnoses_parsed"]].head()

,case_id,diagnoses_parsed
0,e3935ce4-64d3-4a66-ba11-d308b844b410,"[{'synchronous_malignancy': 'No', 'ajcc_pathol..."
1,e3b555aa-7f0a-49c6-9b13-182c61a144c1,"[{'synchronous_malignancy': 'No', 'ajcc_pathol..."
2,17ca61a2-607a-45ff-88fa-ef72e80bf891,"[{'synchronous_malignancy': 'No', 'ajcc_pathol..."
3,7d681cc6-689d-41c8-9e84-e13733089ec9,"[{'synchronous_malignancy': 'No', 'ajcc_pathol..."
4,17f275c1-a0d4-487d-8f02-ea279584b4cd,"[{'synchronous_malignancy': 'No', 'ajcc_pathol..."


## Inspect the number of diagnosis records per case

In [81]:
diag_count_summary = df["diagnoses_parsed"].map(len).value_counts().sort_index()
diag_count_summary

diagnoses_parsed
0      1
1    915
2    141
3     27
4     10
5      1
6      2
7      1
Name: count, dtype: int64

seeing if every attribute is expected

In [82]:
example_idx = df["diagnoses_parsed"].map(len).gt(0).idxmax()
df.loc[example_idx, "diagnoses_parsed"]

[{'synchronous_malignancy': 'No',
  'ajcc_pathologic_stage': 'Stage IIB',
  'days_to_diagnosis': 0,
  'laterality': 'Right',
  'created_datetime': None,
  'last_known_disease_status': None,
  'tissue_or_organ_of_origin': 'Breast, NOS',
  'days_to_last_follow_up': 24.0,
  'age_at_diagnosis': 24053,
  'primary_diagnosis': 'Metaplastic carcinoma, NOS',
  'updated_datetime': '2025-10-24T10:03:04.457495-05:00',
  'prior_malignancy': 'no',
  'year_of_diagnosis': 2012,
  'state': 'released',
  'prior_treatment': 'No',
  'diagnosis_is_primary_disease': True,
  'days_to_last_known_disease_status': None,
  'method_of_diagnosis': 'Cytology',
  'ajcc_staging_system_edition': '7th',
  'ajcc_pathologic_t': 'T3',
  'days_to_recurrence': None,
  'morphology': '8575/3',
  'ajcc_pathologic_n': 'N0',
  'ajcc_pathologic_m': 'M0',
  'submitter_id': 'TCGA-E9-A5FL_diagnosis',
  'classification_of_tumor': 'primary',
  'diagnosis_id': '63d06bf8-70cd-5c28-aee2-6b774b65a672',
  'icd_10_code': 'C50.9',
  'site_of

## Explode and flatten diagnoses

Some patients may have more than one diagnosis record. We first create one row per diagnosis entry, then flatten the nested fields into ordinary columns.

In [83]:
diag_df = (
    df[[
        "case_id",
        "submitter_id",
        "project.project_id",
        "demographic.race",
        "demographic.gender",
        "demographic.ethnicity",
        "diagnoses_parsed"
    ]]
    .explode("diagnoses_parsed")
    .reset_index(drop=True)
)

diag_df["diagnoses_parsed"] = diag_df["diagnoses_parsed"].apply(
    lambda x: x if isinstance(x, dict) else {}
)

diag_flat = pd.json_normalize(diag_df["diagnoses_parsed"])

diag_full = pd.concat(
    [diag_df.drop(columns=["diagnoses_parsed"]), diag_flat],
    axis=1
)

print(diag_full.shape)
diag_full.head()

(1343, 42)


,case_id,submitter_id,project.project_id,demographic.race,demographic.gender,demographic.ethnicity,synchronous_malignancy,ajcc_pathologic_stage,days_to_diagnosis,laterality,...,diagnosis_id,icd_10_code,site_of_resection_or_biopsy,tumor_grade,sites_of_involvement,progression_or_recurrence,metastasis_at_diagnosis,tumor_of_origin,figo_stage,figo_staging_edition_year
0,e3935ce4-64d3-4a66-ba11-d308b844b410,TCGA-E9-A5FL,TCGA-BRCA,white,female,not hispanic or latino,No,Stage IIB,0.0,Right,...,63d06bf8-70cd-5c28-aee2-6b774b65a672,C50.9,"Breast, NOS",None,"[Breast, Right Upper Outer]",None,NaN,NaN,NaN,NaN
1,e3b555aa-7f0a-49c6-9b13-182c61a144c1,TCGA-A2-A1G4,TCGA-BRCA,white,female,not hispanic or latino,No,Stage IIIA,0.0,Right,...,45ee48b9-1dac-54d7-aed5-ce0606ba252e,C50.9,"Breast, NOS",None,"[Breast, Right Lower Outer]",None,No Metastasis,NaN,NaN,NaN
2,e3b555aa-7f0a-49c6-9b13-182c61a144c1,TCGA-A2-A1G4,TCGA-BRCA,white,female,not hispanic or latino,NaN,Stage IIIA,NaN,Left,...,d903dcbd-63d3-4faa-9702-ec8f93d70ba7,NaN,Not Reported,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,17ca61a2-607a-45ff-88fa-ef72e80bf891,TCGA-BH-A0HF,TCGA-BRCA,white,female,not reported,No,Stage IA,0.0,Right,...,14047651-e3d9-5a6a-b989-19b0548d8193,C50.9,"Breast, NOS",None,"[Breast, Right Upper Outer, Breast, NOS]",None,NaN,NaN,NaN,NaN
4,7d681cc6-689d-41c8-9e84-e13733089ec9,TCGA-AR-A1AS,TCGA-BRCA,asian,female,not reported,No,Stage IIB,0.0,Left,...,8737c011-3e1c-5721-920d-ef7e7248413c,C50.9,"Breast, NOS",None,"[Breast, Left Upper Inner]",None,NaN,NaN,NaN,NaN


## Inspect flattened diagnosis columns

In [84]:
diag_full.columns.tolist()

['case_id',
 'submitter_id',
 'project.project_id',
 'demographic.race',
 'demographic.gender',
 'demographic.ethnicity',
 'synchronous_malignancy',
 'ajcc_pathologic_stage',
 'days_to_diagnosis',
 'laterality',
 'created_datetime',
 'last_known_disease_status',
 'tissue_or_organ_of_origin',
 'days_to_last_follow_up',
 'age_at_diagnosis',
 'primary_diagnosis',
 'updated_datetime',
 'prior_malignancy',
 'year_of_diagnosis',
 'state',
 'prior_treatment',
 'diagnosis_is_primary_disease',
 'days_to_last_known_disease_status',
 'method_of_diagnosis',
 'ajcc_staging_system_edition',
 'ajcc_pathologic_t',
 'days_to_recurrence',
 'morphology',
 'ajcc_pathologic_n',
 'ajcc_pathologic_m',
 'submitter_id',
 'classification_of_tumor',
 'diagnosis_id',
 'icd_10_code',
 'site_of_resection_or_biopsy',
 'tumor_grade',
 'sites_of_involvement',
 'progression_or_recurrence',
 'metastasis_at_diagnosis',
 'tumor_of_origin',
 'figo_stage',
 'figo_staging_edition_year']

filter strings for useful columns

In [85]:
sorted([
    c for c in diag_full.columns
    if any(k in c.lower() for k in [
        "day", "vital", "stage", "grade", "diagnosis", "age", "status"
    ])
])

['age_at_diagnosis',
 'ajcc_pathologic_stage',
 'days_to_diagnosis',
 'days_to_last_follow_up',
 'days_to_last_known_disease_status',
 'days_to_recurrence',
 'diagnosis_id',
 'diagnosis_is_primary_disease',
 'figo_stage',
 'last_known_disease_status',
 'metastasis_at_diagnosis',
 'method_of_diagnosis',
 'primary_diagnosis',
 'tumor_grade',
 'year_of_diagnosis']

## Merge top-level death/event fields into the flattened diagnosis table

The diagnosis records now contain follow-up and tumor characteristics, while the raw case table contains top-level demographic death information. We merge these by `case_id` to construct a survival analysis table.

In [86]:
top_demo = df[
    ["case_id", "demographic.vital_status", "demographic.days_to_death"]
].drop_duplicates(subset="case_id").copy()

surv = diag_full.merge(top_demo, on="case_id", how="left")

print(surv.shape)
surv[
    [
        "case_id",
        "demographic.vital_status",
        "demographic.days_to_death",
        "days_to_last_follow_up",
        "primary_diagnosis",
        "ajcc_pathologic_stage",
    ]
].head()

(1343, 44)


,case_id,demographic.vital_status,demographic.days_to_death,days_to_last_follow_up,primary_diagnosis,ajcc_pathologic_stage
0,e3935ce4-64d3-4a66-ba11-d308b844b410,Alive,NaN,24.0,"Metaplastic carcinoma, NOS",Stage IIB
1,e3b555aa-7f0a-49c6-9b13-182c61a144c1,Alive,NaN,595.0,"Infiltrating duct carcinoma, NOS",Stage IIIA
2,e3b555aa-7f0a-49c6-9b13-182c61a144c1,Alive,NaN,NaN,"Infiltrating lobular carcinoma, NOS",Stage IIIA
3,17ca61a2-607a-45ff-88fa-ef72e80bf891,Alive,NaN,727.0,"Infiltrating duct carcinoma, NOS",Stage IA
4,7d681cc6-689d-41c8-9e84-e13733089ec9,Alive,NaN,1150.0,"Infiltrating duct carcinoma, NOS",Stage IIB


take a look at balancing

In [87]:
surv["event"] = (
    surv["demographic.vital_status"]
    .astype(str)
    .str.lower()
    .map({"dead": 1, "alive": 0})
)

surv["event"].value_counts(dropna=False)

event
0.0    1082
1.0     260
NaN       1
Name: count, dtype: int64

In [88]:
surv["time_days"] = np.where(
    surv["event"] == 1,
    surv["demographic.days_to_death"],
    surv["days_to_last_follow_up"]
)

surv[
    [
        "demographic.vital_status",
        "event",
        "demographic.days_to_death",
        "days_to_last_follow_up",
        "time_days",
    ]
].head(10)

,demographic.vital_status,event,demographic.days_to_death,days_to_last_follow_up,time_days
0,Alive,0.0,NaN,24.0,24.0
1,Alive,0.0,NaN,595.0,595.0
2,Alive,0.0,NaN,NaN,NaN
3,Alive,0.0,NaN,727.0,727.0
4,Alive,0.0,NaN,1150.0,1150.0
5,Alive,0.0,NaN,1728.0,1728.0
6,Alive,0.0,NaN,996.0,996.0
7,Alive,0.0,NaN,323.0,323.0
8,Alive,0.0,NaN,554.0,554.0
9,Alive,0.0,NaN,1247.0,1247.0


In [89]:
surv["age_at_diagnosis_years"] = surv["age_at_diagnosis"] / 365.25

surv[["age_at_diagnosis", "age_at_diagnosis_years"]].head()

,age_at_diagnosis,age_at_diagnosis_years
0,24053.0,65.853525
1,25966.0,71.091034
2,NaN,NaN
3,28233.0,77.297741
4,20028.0,54.833676


In [90]:
surv_valid = surv.loc[
    surv["event"].notna() &
    surv["time_days"].notna() &
    (surv["time_days"] >= 0)
].copy()

print("Rows before filtering:", surv.shape[0])
print("Rows after filtering :", surv_valid.shape[0])

Rows before filtering: 1343
Rows after filtering : 1203


In [91]:
surv_valid["nonmissing_score"] = surv_valid[
    [
        "primary_diagnosis",
        "ajcc_pathologic_stage",
        "tumor_grade",
        "age_at_diagnosis_years",
        "time_days",
        "event",
    ]
].notna().sum(axis=1)

surv_case_level = (
    surv_valid
    .sort_values(["case_id", "nonmissing_score"], ascending=[True, False])
    .drop_duplicates(subset="case_id", keep="first")
    .reset_index(drop=True)
)

print("Final dataset shape:", surv_case_level.shape)

Final dataset shape: (1095, 48)


In [92]:
analysis_df = surv_case_level[
    [
        "case_id",
        "primary_diagnosis",
        "ajcc_pathologic_stage",
        "tumor_grade",
        "demographic.gender",
        "demographic.race",
        "age_at_diagnosis_years",
        "event",
        "time_days",
    ]
].copy()

analysis_df.head()

,case_id,primary_diagnosis,ajcc_pathologic_stage,tumor_grade,demographic.gender,demographic.race,age_at_diagnosis_years,event,time_days
0,001cef41-ff86-4d3f-a140-a647ac4b10a1,"Infiltrating duct carcinoma, NOS",Stage IA,None,female,white,60.996578,0.0,337.0
1,0045349c-69d9-4306-a403-c9c1fa836644,Adenoid cystic carcinoma,Stage I,None,female,white,70.726899,0.0,259.0
2,00807dae-9f4a-4fd1-aac2-82eb11bf2afb,Apocrine adenocarcinoma,Stage IIB,None,female,white,50.225873,0.0,3102.0
3,00a2d166-78c9-4687-a195-3d6315c27574,"Infiltrating duct carcinoma, NOS",Stage IIA,None,female,white,56.709103,0.0,5.0
4,00b11ca8-8540-4a3d-b602-ec754b00230b,"Lobular carcinoma, NOS",Stage IA,None,female,white,61.593429,0.0,759.0


In [93]:
analysis_df["event"].value_counts(dropna=False)

event
0.0    944
1.0    151
Name: count, dtype: int64

In [94]:
analysis_df["time_days"].describe()

count    1095.000000
mean     1234.346119
std      1191.475805
min         0.000000
25%       446.000000
50%       825.000000
75%      1673.000000
max      8605.000000
Name: time_days, dtype: float64

In [95]:
analysis_df["stage_simple"] = (
    analysis_df["ajcc_pathologic_stage"]
    .astype(str)
    .str.extract(r"(Stage\s+[IVX]+)")
)

analysis_df.loc[
    analysis_df["stage_simple"] == "Stage X",
    "stage_simple"
] = np.nan

analysis_df["stage_simple"].value_counts(dropna=False)

stage_simple
Stage II     620
Stage III    248
Stage I      183
NaN           24
Stage IV      20
Name: count, dtype: int64

In [96]:
analysis_df = analysis_df.drop(columns=["tumor_grade"]) # since it doesnt matter much for survival

In [97]:
analysis_df.isna().sum()

case_id                    0
primary_diagnosis          0
ajcc_pathologic_stage     11
demographic.gender         0
demographic.race           0
age_at_diagnosis_years    15
event                      0
time_days                  0
stage_simple              24
dtype: int64

In [98]:
analysis_df["primary_diagnosis"].value_counts().head(15)

primary_diagnosis
Infiltrating duct carcinoma, NOS                            776
Lobular carcinoma, NOS                                      201
Infiltrating duct and lobular carcinoma                      28
Infiltrating duct mixed with other types of carcinoma        19
Mucinous adenocarcinoma                                      16
Metaplastic carcinoma, NOS                                   14
Infiltrating lobular mixed with other types of carcinoma      7
Intraductal papillary adenocarcinoma with invasion            6
Medullary carcinoma, NOS                                      5
Invasive micropapillary carcinoma                             4
Pleomorphic carcinoma                                         3
Paget disease and infiltrating duct carcinoma of breast       3
Phyllodes tumor, malignant                                    2
Papillary carcinoma, NOS                                      2
Carcinoma, NOS                                                1
Name: count, dtype: in

In [99]:
analysis_df.to_csv(
    processed_dir / "tcga_brca_survival_cleaned.csv",
    index=False
)

print("Saved cleaned dataset.")

Saved cleaned dataset.
